In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = str(project_root / "src")

# Portfolio Environment (Phase 1)

In this phase, the reinforcement learning environment is initialized and validated.

The objectives of this phase are:

1. Load the prepared environment dataset.
2. Initialize the Gymnasium environment.
3. Reset the environment.
4. Verify the observation shape.
5. Inspect the generated state before implementing the interaction logic.

At this stage, the environment only provides observations. No actions, rewards or transitions are implemented yet.

In [2]:
from environment import PortfolioEnv
from feature_engineering import FeatureEngineer

In [3]:
engineer = FeatureEngineer()

environment_data = engineer.load_dataset()

In [4]:
env = PortfolioEnv(
    dataset=environment_data,
    portfolio="Conservative",
)

In [5]:
state, info = env.reset()

In [6]:
print(state.shape)

(16, 10)


In [7]:
state

array([[ 2.01977062e-04,  1.08605229e-04,  5.85772941e-05,
         1.04058097e-04,  5.80999294e-05,  6.30417053e-05,
         8.64713220e-05,  4.90869643e-05,  7.66716403e-05,
         4.10448411e-05],
       [ 1.08605229e-04,  2.57301785e-04,  7.38914387e-05,
         2.06865254e-04,  8.70938893e-05,  8.49138160e-05,
         1.07264968e-04,  7.24815327e-05,  1.06775013e-04,
         6.26285473e-05],
       [ 5.85772941e-05,  7.38914387e-05,  6.73242539e-05,
         7.75416993e-05,  4.39265714e-05,  3.67774373e-05,
         5.62584682e-05,  4.08536398e-05,  4.63329015e-05,
         3.07772170e-05],
       [ 1.04058097e-04,  2.06865254e-04,  7.75416993e-05,
         3.74317809e-04,  8.11569626e-05,  6.38507699e-05,
         1.18538475e-04,  7.32093104e-05,  1.34146205e-04,
         5.23087801e-05],
       [ 5.80999294e-05,  8.70938893e-05,  4.39265714e-05,
         8.11569626e-05,  9.78637399e-05,  4.76356945e-05,
         5.80158812e-05,  4.96355133e-05,  5.65336704e-05,
         4.

In [8]:
env.render()

Portfolio : Conservative
Step      : 0
Date      : 2011-01-04 00:00:00


# Portfolio Environment (Phase 2)

In this phase, the interaction logic of the reinforcement learning environment is validated.

The objectives of this phase are:

1. Execute random portfolio allocation actions.
2. Compute daily asset returns.
3. Compute portfolio return.
4. Update portfolio value.
5. Verify reward calculation.
6. Verify episode progression.

At this stage, transaction costs are intentionally ignored. The reward is equal to the daily portfolio return.

In [9]:
import numpy as np

from environment import PortfolioEnv
from feature_engineering import FeatureEngineer

In [10]:
engineer = FeatureEngineer()

environment_data = engineer.load_dataset()

env = PortfolioEnv(
    environment_data,
    "Conservative",
)

In [11]:
state, info = env.reset()

In [12]:
action = np.ones(10)

action /= action.sum()

In [13]:
next_state, reward, terminated, truncated, info = env.step(action)

print("Reward :", reward)
print("Portfolio Value :", info["portfolio_value"])
print("Date :", info["date"])

Reward : 0.0012603188694920394
Portfolio Value : 1.001260318869492
Date : 2011-01-05 00:00:00


In [14]:
env.render()

Portfolio : Conservative
Step      : 1
Date      : 2011-01-05 00:00:00


# Reward Function Validation (Phase 3)

This phase validates the complete reward mechanism of the reinforcement learning environment.

The objectives are:

1. Execute portfolio allocation actions.
2. Compute daily portfolio return.
3. Compute transaction cost.
4. Compute the final reward.
5. Update portfolio value.
6. Verify that transaction costs are correctly applied whenever portfolio weights change.

In [15]:
import numpy as np

from environment import PortfolioEnv
from feature_engineering import FeatureEngineer

In [16]:
engineer = FeatureEngineer()

environment_data = engineer.load_dataset()

env = PortfolioEnv(
    environment_data,
    "Conservative",
)

In [17]:
state, info = env.reset()

In [18]:
action = np.array(
    [
        0.30,
        0.10,
        0.10,
        0.10,
        0.10,
        0.05,
        0.05,
        0.05,
        0.10,
        0.05,
    ],
    dtype=np.float32,
)

In [19]:
next_state, reward, terminated, truncated, info = env.step(action)

print(f"Reward            : {reward:.6f}")

print(f"Portfolio Return  : {info['portfolio_return']:.6f}")

print(f"Transaction Cost  : {info['transaction_cost']:.6f}")

print(f"Portfolio Value   : {info['portfolio_value']:.6f}")

Reward            : 0.001110
Portfolio Return  : 0.001610
Transaction Cost  : 0.000500
Portfolio Value   : 1.001110


# Environment Validation (Phase 4)

This phase verifies that the custom Gymnasium environment complies with the Stable-Baselines3 API.

The environment checker validates:

- Observation Space
- Action Space
- Reset API
- Step API
- Returned data types
- Episode termination logic

Passing this validation ensures that the environment is fully compatible with Stable-Baselines3 before training the PPO agent.

In [21]:
from stable_baselines3.common.env_checker import check_env

from environment import PortfolioEnv
from feature_engineering import FeatureEngineer

In [22]:
engineer = FeatureEngineer()

environment_data = engineer.load_dataset()

env = PortfolioEnv(
    environment_data,
    "Conservative",
)

In [23]:
check_env(env)

f:\SBU\RL\DRL_Final_Project\.venv\Lib\site-packages\stable_baselines3\common\env_checker.py:324: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(
f:\SBU\RL\DRL_Final_Project\.venv\Lib\site-packages\stable_baselines3\common\env_checker.py:515: UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf. https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
  warnings.warn(
